# SmartStock ML Phase 6: Exploratory Analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebass369/smartstock-ml/blob/main/notebooks/phase_6_exploratory_analysis.ipynb)

## 1. Project and Privacy

This notebook tells a short, reproducible story about synthetic demand, inventory, stockouts, and full-pack ordering decisions. It uses only the nine approved public product aliases and deterministic records generated by SmartStock ML.

All values and charts are synthetic. They are not sales records, operational guidance, ordering guidance, or food-safety guidance. The baseline policy is not a forecasting model, an optimization system, or evidence of real-world improvement.

## 2. Reproducible Colab Setup

A fresh Colab runtime clones the public repository at the approved immutable source revision and installs only the project plus pandas and matplotlib. Network access is used only for that setup. Local uploads and Google Drive are not required.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/sebass369/smartstock-ml.git"
SOURCE_REVISION = "8d5b9f9a8b5ee9398b04e6fa2713e4def89d13d4"

try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    project_root = Path("/content/smartstock-ml")
    if project_root.exists():
        raise RuntimeError(
            "Expected a fresh Colab runtime, but the checkout already exists. "
            "Restart the runtime and run all cells again."
        )
    subprocess.run(
        ["git", "clone", "--quiet", REPOSITORY_URL, str(project_root)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(project_root), "checkout", "--quiet", SOURCE_REVISION],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", str(project_root)],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "pandas>=2.2,<4.0",
            "matplotlib>=3.8,<4.0",
        ],
        check=True,
    )
else:
    project_root = Path.cwd()
    if not (project_root / "PROJECT.md").is_file():
        raise RuntimeError(
            "Run this notebook from the SmartStock ML repository root."
        )

resolved_revision = subprocess.check_output(
    ["git", "-C", str(project_root), "rev-parse", "HEAD"],
    text=True,
).strip()
if IN_COLAB and resolved_revision != SOURCE_REVISION:
    raise RuntimeError("The checkout does not match the approved source revision.")

print(f"Project root: {project_root}")
print(f"Source revision: {resolved_revision}")

## 3. Load Configuration and Generate Records

The notebook calls the validated project APIs directly. Demand generation, FIFO inventory transitions, expiration, stockouts, full-pack rounding, ordering recommendations, and scenario comparison remain in `src/smartstock/`. No generated CSV is needed.

In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

from smartstock.config import (
    ENGLISH_WEEKDAYS,
    load_delivery_config,
    load_generation_config,
    load_inventory_config,
    load_ordering_config,
    load_products_config,
)
from smartstock.generator import generate_daily_records
from smartstock.inventory import simulate_inventory
from smartstock.ordering import (
    SCENARIO_METRICS,
    compare_scenarios,
    simulate_policy_inventory,
)

DEFAULT_SEED = 42
CONFIG_ROOT = project_root / "config"

products_config = load_products_config(CONFIG_ROOT / "products.yaml")
delivery_config = load_delivery_config(CONFIG_ROOT / "delivery.yaml")
generation_config = load_generation_config(CONFIG_ROOT / "generation.yaml")
inventory_config = load_inventory_config(
    CONFIG_ROOT / "inventory.yaml",
    products_config,
)
ordering_config = load_ordering_config(
    CONFIG_ROOT / "ordering.yaml",
    products_config,
    delivery_config,
)

product_order = [product["product_id"] for product in products_config["products"]]
high_demand_days_by_product = {
    product["product_id"]: set(product["high_demand_days"])
    for product in products_config["products"]
}
product_colors = dict(
    zip(
        product_order,
        plt.get_cmap("tab10").colors[: len(product_order)],
        strict=True,
    )
)
weekday_order = list(ENGLISH_WEEKDAYS)

demand_records = generate_daily_records(
    products_config,
    delivery_config,
    generation_config,
    seed=DEFAULT_SEED,
)
fixed_records = simulate_inventory(
    demand_records,
    products_config,
    delivery_config,
    inventory_config,
)
policy_records, recommendation_records = simulate_policy_inventory(
    demand_records,
    products_config,
    delivery_config,
    inventory_config,
    ordering_config,
)

cycle_length_days = delivery_config["delivery"]["cycle_length_days"]
evaluation_start_date = (
    date.fromisoformat(demand_records[0]["date"])
    + timedelta(days=cycle_length_days)
)
scenario_comparison = compare_scenarios(
    fixed_records,
    policy_records,
    products_config,
    evaluation_start_date,
)

## 4. Validate and Inspect the Synthetic Dataset

The generated record count, product scope, date range, and seed form a compact reproducibility check. Product order is preserved from `products.yaml`.

In [ ]:
demand_frame = pd.DataFrame(demand_records)
fixed_frame = pd.DataFrame(fixed_records)
policy_frame = pd.DataFrame(policy_records)
recommendation_frame = pd.DataFrame(recommendation_records)

for frame in (demand_frame, fixed_frame, policy_frame):
    frame["date"] = pd.to_datetime(frame["date"])
    frame["product_id"] = pd.Categorical(
        frame["product_id"],
        categories=product_order,
        ordered=True,
    )

recommendation_frame["delivery_date"] = pd.to_datetime(
    recommendation_frame["delivery_date"]
)
recommendation_frame["product_id"] = pd.Categorical(
    recommendation_frame["product_id"],
    categories=product_order,
    ordered=True,
)

dataset_overview = pd.DataFrame(
    [
        {
            "record_count": len(demand_frame),
            "product_count": demand_frame["product_id"].nunique(),
            "start_date": demand_frame["date"].min().date(),
            "end_date": demand_frame["date"].max().date(),
            "seed": DEFAULT_SEED,
        }
    ]
)

assert dataset_overview.loc[0, "record_count"] == 504
assert dataset_overview.loc[0, "product_count"] == 9
assert len(fixed_frame) == len(policy_frame) == 504
assert len(recommendation_frame) == 27
dataset_overview

## 5. Explore Synthetic Demand

Total and mean daily demand provide a simple product-level comparison. These values come from the configured synthetic ranges and seeded random generator; they are not observations of customer behavior.

In [ ]:
demand_summary = (
    demand_frame.groupby("product_id", observed=True, sort=False)
    .agg(
        total_demand_units=("demand_units", "sum"),
        mean_daily_demand_units=("demand_units", "mean"),
    )
    .reindex(product_order)
    .reset_index()
)
demand_summary["mean_daily_demand_units"] = demand_summary["mean_daily_demand_units"].round(2)
demand_summary

In [ ]:
plot_data = demand_summary.iloc[::-1]
fig, axis = plt.subplots(figsize=(10, 5.5))
bars = axis.barh(
    plot_data["product_id"].astype(str),
    plot_data["total_demand_units"],
    color=[product_colors[product_id] for product_id in plot_data["product_id"]],
)
axis.bar_label(bars, padding=3)
axis.set_title("Total Synthetic Demand by Product")
axis.set_xlabel("Synthetic demand units over 56 days")
axis.set_ylabel("Approved public product alias")
axis.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

Configured high-demand weekdays add one synthetic unit. The small multiples make all nine products readable without placing 63 bars in one panel. Products without configured high-demand weekdays still vary because their base demand is randomly generated from a fixed seeded range.

In [ ]:
weekday_demand = (
    demand_frame.groupby(["product_id", "weekday"], observed=True)["demand_units"]
    .mean()
    .unstack("weekday")
    .reindex(index=product_order, columns=weekday_order)
)

fig, axes = plt.subplots(3, 3, figsize=(15, 11), sharey=True)
for axis, product_id in zip(axes.flat, product_order, strict=True):
    values = weekday_demand.loc[product_id]
    bars = axis.bar(
        weekday_order,
        values,
        color=product_colors[product_id],
        edgecolor="black",
        linewidth=0.5,
    )
    for bar, weekday in zip(bars, weekday_order, strict=True):
        if weekday in high_demand_days_by_product[product_id]:
            bar.set_hatch("///")
    axis.set_title(product_id)
    axis.tick_params(axis="x", rotation=45)
    axis.grid(axis="y", alpha=0.25)
for axis in axes[:, 0]:
    axis.set_ylabel("Mean synthetic demand units")
fig.legend(
    handles=[
        Patch(
            facecolor="white",
            edgecolor="black",
            hatch="///",
            label="Configured high-demand weekday",
        )
    ],
    loc="upper center",
)
fig.suptitle("Mean Synthetic Demand by Weekday and Product", y=1.03, fontsize=15)
fig.tight_layout()
plt.show()

## 6. Explore Inventory and Stockouts

The fixed scenario repeats configured delivery pack counts. After the shared 14-day warm-up, the policy scenario replaces fixed deliveries with recommendations based on the previous completed cycle. Lower ending inventory is not automatically better: the policy scenario also produces synthetic unmet demand.

In [ ]:
delivery_dates = (
    fixed_frame.loc[fixed_frame["delivery_event"], "date"]
    .drop_duplicates()
    .sort_values()
)
fig, axes = plt.subplots(3, 3, figsize=(16, 11), sharex=True)
for axis, product_id in zip(axes.flat, product_order, strict=True):
    fixed_product = fixed_frame.loc[fixed_frame["product_id"] == product_id]
    policy_product = policy_frame.loc[policy_frame["product_id"] == product_id]
    axis.plot(
        fixed_product["date"],
        fixed_product["ending_inventory_units"],
        label="Fixed scenario",
        color="#1f77b4",
        linewidth=1.8,
    )
    axis.plot(
        policy_product["date"],
        policy_product["ending_inventory_units"],
        label="Policy scenario",
        color="#d62728",
        linestyle="--",
        linewidth=1.8,
    )
    for delivery_date in delivery_dates:
        axis.axvline(delivery_date, color="#777777", alpha=0.2, linewidth=1)
    axis.set_title(product_id)
    axis.grid(alpha=0.2)
    axis.tick_params(axis="x", rotation=35)
for axis in axes[:, 0]:
    axis.set_ylabel("Ending inventory units")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
fig.suptitle("Synthetic Ending Inventory by Product and Scenario", y=1.02, fontsize=15)
fig.tight_layout()
plt.show()

In [ ]:
policy_stockouts = (
    policy_frame.groupby("product_id", observed=True, sort=False)
    .agg(
        unmet_demand_units=("unmet_demand_units", "sum"),
        stockout_event_count=("stockout_event", "sum"),
    )
    .reindex(product_order)
    .reset_index()
)
plot_data = policy_stockouts.iloc[::-1]
fig, axis = plt.subplots(figsize=(10, 5.5))
bars = axis.barh(
    plot_data["product_id"].astype(str),
    plot_data["unmet_demand_units"],
    color=[product_colors[product_id] for product_id in plot_data["product_id"]],
)
for bar, stockout_count in zip(bars, plot_data["stockout_event_count"], strict=True):
    axis.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{int(stockout_count)} stockout events",
        va="center",
    )
axis.set_title("Synthetic Unmet Demand in the Policy Scenario")
axis.set_xlabel("Unmet synthetic demand units over 56 days")
axis.set_ylabel("Approved public product alias")
axis.grid(axis="x", alpha=0.25)
axis.margins(x=0.3)
fig.tight_layout()
plt.show()

## 7. Review Ordering Recommendations

Recommendations occur on January 21, February 4, and February 18 after the shared warm-up cycle. Quantities are rounded to full packs by the existing Phase 4B policy.

In [ ]:
recommendation_summary = (
    recommendation_frame.groupby("product_id", observed=True, sort=False)
    .agg(
        recommendation_count=("delivery_date", "count"),
        recommended_order_packs=("recommended_order_packs", "sum"),
        recommended_order_units=("recommended_order_units", "sum"),
    )
    .reindex(product_order)
    .reset_index()
)
recommendation_summary

In [ ]:
recommendation_matrix = (
    recommendation_frame.pivot(
        index="product_id",
        columns="delivery_date",
        values="recommended_order_packs",
    )
    .reindex(product_order)
)
fig, axis = plt.subplots(figsize=(8, 7))
image = axis.imshow(recommendation_matrix.to_numpy(), cmap="Blues", aspect="auto")
axis.set_xticks(
    range(len(recommendation_matrix.columns)),
    [value.strftime("%Y-%m-%d") for value in recommendation_matrix.columns],
    rotation=30,
    ha="right",
)
axis.set_yticks(range(len(product_order)), product_order)
axis.set_xlabel("Synthetic delivery date")
axis.set_ylabel("Approved public product alias")
axis.set_title("Recommended Full Packs by Product and Delivery Date")
threshold = recommendation_matrix.to_numpy().max() / 2
for row_index in range(recommendation_matrix.shape[0]):
    for column_index in range(recommendation_matrix.shape[1]):
        value = int(recommendation_matrix.iloc[row_index, column_index])
        axis.text(
            column_index,
            row_index,
            str(value),
            ha="center",
            va="center",
            color="white" if value > threshold else "black",
        )
fig.colorbar(image, ax=axis, label="Recommended packs")
fig.tight_layout()
plt.show()

## 8. Compare Fixed and Policy Scenarios

The comparison begins on January 21, after the shared warm-up cycle. Each difference is policy minus fixed. Metrics are kept separate because lower values are not uniformly better: fewer deliveries may coexist with more unmet demand, and lower ending inventory may reduce buffer.

In [ ]:
fixed_overall = scenario_comparison["fixed"]["overall"]
policy_overall = scenario_comparison["policy"]["overall"]
difference_overall = scenario_comparison["difference"]["overall"]
overall_comparison = pd.DataFrame(
    [
        {
            "metric": metric,
            "fixed": fixed_overall[metric],
            "policy": policy_overall[metric],
            "policy_minus_fixed": difference_overall[metric],
        }
        for metric in SCENARIO_METRICS
    ]
)
overall_comparison

In [ ]:
difference_by_product = scenario_comparison["difference"]["by_product"]
product_differences = pd.DataFrame(
    [
        {"product_id": product_id, **difference_by_product[product_id]}
        for product_id in product_order
    ]
)
product_differences

The default fixed scenario fulfills all synthetic demand in the evaluation window. The policy scenario delivers fewer packs and finishes with less inventory, but it also produces synthetic unmet demand and stockout events. Neither scenario is proven optimal.

## 9. Limitations and Phase 7 Handoff

- Every record is synthetic and deterministic for the validated configuration and seed 42.
- The 56-day default simulation is shorter than every approved unopened shelf life, so default expiration and waste are zero by design. Focused unit tests cover shorter expiration boundaries.
- Some high-demand-day and risk fields remain explicitly provisional in `products.yaml`.
- The baseline uses the previous completed 14-day demand cycle, zero default safety stock, and a synthetic zero-day lead time.
- The comparison covers one constructed scenario and cannot establish real-world effectiveness or optimality.
- This notebook performs descriptive analysis only. Forecasting, feature design, training, prediction, and model evaluation remain separate Phase 7 work.
- Future predictive work must preserve the documented leakage boundaries and must not use same-row or future outcomes as model inputs.